# Epithelial cells by region: AI vs Otsu

Count **Tumor epithelial cells** and **Normal epithelial cells** in each paired
`*_Otsu.zarr` / `*_AI.zarr` project. Region names such as `Lesion 1` and `Lung 2`
are grouped by their first word. A cell's centroid determines its region;
**Lesion takes priority over Lung**, so overlapping cells count only in Lesion.
Cells outside both regions are excluded.

A cell belongs to a type when at least 50% of its segmentation pixels are covered
by that type mask. A missing type mask counts as zero (gating may omit empty types).

The Fisher value is the existing **Fisher separation score J**, calculated separately
for each cell type and method using its fraction of **all segmented cells** in each region:

$$J = \frac{(p_{Lesion} - p_{Lung})^2}
{p_{Lesion}(1-p_{Lesion}) + p_{Lung}(1-p_{Lung})}.$$

Equal proportions give J = 0; perfect separation gives infinity; an empty region
makes J undefined (NaN). The sample is the leading number in the project name
(`20231208` or `20231214`). Counts are pooled over its cases **before** calculating J.
The **Both together** result pools both samples before calculating J. J is unsigned.
Results are displayed only in this notebook; no files are written.

In [4]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import ndimage
from skimage.draw import polygon

# Run from either the repository root or its notebooks directory.
REPO_ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "opal_studio" / "project_io.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from opal_studio import project_io

PROJECT_ROOT = REPO_ROOT / "Projects"
METHODS = ("Otsu", "AI")
SEGMENTATION = "StarDist_Expand"
CELL_TYPES = ("Tumor epithelial cells", "Normal epithelial cells")
REGIONS = ("Lesion", "Lung")
MEMBERSHIP_FRAC = 0.5

In [5]:
def discover_pairs(root):
    """Find complete Otsu/AI project pairs, retaining the full project stem."""
    pairs = []
    for otsu in sorted(root.glob("*_Otsu.zarr")):
        stem = otsu.name.removesuffix("_Otsu.zarr")
        stores = {method: root / f"{stem}_{method}.zarr" for method in METHODS}
        if all((store / "zarr.json").is_file() for store in stores.values()):
            pairs.append((stem, stores))
        else:
            print(f"Skipping incomplete pair: {stem}")
    if not pairs:
        raise ValueError(f"No complete Otsu/AI project pairs in {root}")
    return pairs


def read_masks(store):
    """Load only the segmentation, the two type masks, and Lesion/Lung polygons."""
    masks = {}
    for element in sorted((store / "labels").iterdir()):
        key = project_io._read_element_key(element / "zarr.json", element.name)
        if key in (SEGMENTATION, *CELL_TYPES):
            array = project_io._read_v3_array(element / "0")
            masks[key] = array[0] if array.ndim == 3 else array
    if SEGMENTATION not in masks:
        raise KeyError(f"{store.name}: missing segmentation {SEGMENTATION!r}")
    shape = masks[SEGMENTATION].shape
    if len(shape) != 2 or any(mask.shape != shape for mask in masks.values()):
        raise ValueError(f"{store.name}: masks must be matching 2D arrays")

    regions = {name: np.zeros(shape, dtype=bool) for name in REGIONS}
    found = set()
    for element in sorted((store / "shapes").iterdir()):
        key = project_io._read_element_key(element / "zarr.json", element.name)
        group = key.split()[0].rstrip("?") if key.split() else ""
        if group not in regions:
            continue
        found.add(group)
        rings_by_id = project_io._read_shapes_parquet(element / "shapes.parquet")
        for rings in rings_by_id.values():
            for ring in rings:
                xy = np.asarray(ring, dtype=float)
                if len(xy) >= 3:
                    rr, cc = polygon(xy[:, 1], xy[:, 0], shape=shape)
                    regions[group][rr, cc] = True
    if found != set(REGIONS):
        raise ValueError(f"{store.name}: missing region polygons: {set(REGIONS) - found}")

    # Subtract Lesion from Lung before assigning cells by centroid.
    regions["Lung"] &= ~regions["Lesion"]
    return masks, regions


def fisher_j(lesion_count, lung_count, lesion_total, lung_total):
    """Binary-feature Fisher separation, using all cells as each denominator."""
    if lesion_total == 0 or lung_total == 0:
        return np.nan
    p = lesion_count / lesion_total
    q = lung_count / lung_total
    numerator = (p - q) ** 2
    denominator = p * (1 - p) + q * (1 - q)
    if denominator == 0:
        return 0.0 if numerator == 0 else np.inf
    return numerator / denominator


def count_project(store):
    """Return regional counts for the two epithelial cell types."""
    masks, regions = read_masks(store)
    cells = masks[SEGMENTATION]
    ids = np.unique(cells)
    ids = ids[ids > 0]
    if not ids.size:
        raise ValueError(f"{store.name}: segmentation has no cells")

    centres = np.asarray(ndimage.center_of_mass(cells > 0, cells, ids)).astype(int)
    inside = {name: mask[centres[:, 0], centres[:, 1]] for name, mask in regions.items()}
    totals = {name: int(hit.sum()) for name, hit in inside.items()}
    sizes = np.bincount(cells.ravel())
    rows = []
    for cell_type in CELL_TYPES:
        member = np.zeros(ids.size, dtype=bool)
        if cell_type in masks:
            covered = np.bincount(cells[masks[cell_type] > 0], minlength=sizes.size)
            member = covered[ids] / sizes[ids] >= MEMBERSHIP_FRAC
        counts = {name: int((member & inside[name]).sum()) for name in REGIONS}
        rows.append({
            "cell_type": cell_type,
            **counts,
            **{f"total {name}": totals[name] for name in REGIONS},
        })
    return rows

## Read the project counts

Cases are grouped by the leading sample number in their project names.
`total Lesion` and `total Lung` include every segmented cell in each region
and provide the denominators for Fisher J.

In [6]:
pairs = discover_pairs(PROJECT_ROOT)
records = []
for stem, stores in pairs:
    print(f"Reading {stem}")
    for method, store in stores.items():
        for row in count_project(store):
            records.append({"sample": stem.split("_")[0], "case": stem,
                            "method": method, **row})

results = pd.DataFrame(records)

Reading 20231208_712Lung_T_20231208-162603-634659_Q001
Reading 20231208_712Lung_T_20231208-162603-634659_Q002
Reading 20231208_712Lung_T_20231208-162603-634659_Q003
Reading 20231208_712Lung_T_20231208-162603-634659_Q004
Reading 20231214_714Lung_E_3_20231214-153352-220044_Q001
Reading 20231214_714Lung_E_3_20231214-153352-220044_Q002
Reading 20231214_714Lung_E_3_20231214-153352-220044_Q003
Reading 20231214_714Lung_E_3_20231214-153352-220044_Q004
Reading 20231214_714Lung_E_3_20231214-153352-220044_Q005
Reading 20231214_714Lung_E_3_20231214-153352-220044_Q006
Reading 20231214_714Lung_E_3_20231214-153352-220044_Q007
Reading 20231214_714Lung_E_3_20231214-153352-220044_Q008


## Counts and Fisher J per sample, and both together

Each row uses pooled cell counts for one sample, method, and cell type.
**Both together** pools the counts from both samples. Fisher J is calculated
from these pooled counts, rather than averaging case or sample scores.

In [7]:
count_columns = ["Lesion", "Lung", "total Lesion", "total Lung"]
per_sample = results.groupby(["sample", "method", "cell_type"])[count_columns].sum()
combined = results.groupby(["method", "cell_type"])[count_columns].sum()
combined = pd.concat({"Both together": combined}, names=["sample"])
summary = pd.concat([per_sample, combined])
summary["Fisher J"] = [
    fisher_j(lesion, lung, total_lesion, total_lung)
    for lesion, lung, total_lesion, total_lung in summary[count_columns].itertuples(
        index=False, name=None
    )
]

for sample in summary.index.get_level_values("sample").unique():
    print(f"\n{sample}")
    display(summary.loc[sample].reindex(
        pd.MultiIndex.from_product([METHODS, CELL_TYPES], names=["method", "cell_type"])
    ).style.format({"Fisher J": "{:.6f}"}, na_rep="NaN"))


20231208



20231214



Both together


## Youden's J: discrimination in the expected direction

Using anatomical region as the reference, Youden's J is sensitivity + specificity - 1:

- **Tumor epithelial cells:** tumor fraction in Lesion minus tumor fraction in Lung.
- **Normal epithelial cells:** normal fraction in Lung minus normal fraction in Lesion.

Each fraction uses **all segmented cells in that region**. Scores range from -1 to +1;
higher means stronger discrimination in the expected direction. Zero means equal
regional proportions; negative values indicate concentration in the opposite region.
An empty region gives NaN. Rare types can have small scores even when localized.
This measures regional discrimination, rather than independently validated cell-type accuracy.

The table uses the same pooled counts as the Fisher results above, including **Both together**.


In [ ]:
lesion_fraction = summary["Lesion"] / summary["total Lesion"].replace(0, np.nan)
lung_fraction = summary["Lung"] / summary["total Lung"].replace(0, np.nan)
direction = summary.index.get_level_values("cell_type").map({
    "Tumor epithelial cells": 1,
    "Normal epithelial cells": -1,
})
youden = ((lesion_fraction - lung_fraction) * direction).rename("Youden J")
youden_table = youden.unstack("method").reindex(
    index=pd.MultiIndex.from_product(
        [summary.index.get_level_values("sample").unique(), CELL_TYPES],
        names=["sample", "cell_type"],
    ),
    columns=METHODS,
)
display(youden_table.style.format("{:+.6f}", na_rep="NaN"))
